In [ ]:
import pandas as pd 
import numpy as np

def get_landscape_data(agregate_runs=True):
    df_merged = None
    for seed in range(1, 101):
        df_ELA = pd.read_csv(f"./data/raw_data/ELA_data/ELA_LHS_20d/featELA_lhs_20d_1000n_seed{seed}.csv")
        df_ELA.index = df_ELA.apply(lambda row: f'{row.func.split("F")[1]}{row.ins.split("ins")[1]}{row.seed.split("seed")[1]}', axis=1)
        df_ELA = df_ELA.drop(['func', 'ins', 'seed'], axis = 1)
        variable = '\d+_([1-9]|10)_\d+'
        df_ELA = df_ELA[df_ELA.index.str.contains(fr'{variable}', regex=True)]
        df_merged = df_ELA if df_merged is None else pd.concat([df_merged, df_ELA])
    print(df_merged.shape)
    if agregate_runs:
        df_avg = pd.DataFrame(columns=df_merged.columns)
        for fid in range(1, 25):
            for iid in range(1, 11):
                variable = f'{fid}_{iid}.*'
                new_id = f'{fid}_{iid}'
                df_sub = df_merged[df_merged.index.str.contains(fr'\b{variable}\b', regex=True)]
                avg = np.median(df_sub, axis = 0)
                df_avg.loc[new_id] = avg
        print(df_avg.shape)
        df_avg.to_csv(f'./data/processed_data/landscape_data/ELA_LHS_20D_1000n_100seed_all.csv')                    


def get_ng_performance_data(algos, dims, budgets, runs, agregate_runs=True, log_target=True):
    log_flag = "_log" if log_target else ""
    for algo in algos:
        print(algo)
        for dim in dims: 
            for budget in budgets:
                print(budget)
                df_merged = None
                for fid in range(1, 25):
                    for iid in range(1, 11):
                        df = pd.read_csv(f'./data/raw_data/performance_data/{dim}D/{algo}_F{fid}_I{iid}.csv', index_col=0)
                        df = df[df['runtime']==budget]
                        df.index = df.apply(lambda row: f'{row.funcId}_{10 if row.IID==0 else row.IID}_{row.DIM}_{row.runtime}_{row.run}', axis=1)
                        df = df.drop(['DIM', 'funcId', 'ID', 'runtime', 'run', 'IID'], axis=1)
                        df.columns = ['target']
                        df_merged = df if df_merged is None else pd.concat([df_merged, df])
                # print("merged")
                # print(df_merged.shape)
                if log_target:
                    df_merged['target'] = df_merged.apply(lambda row: np.log10(10**(-8)) if row.target == 0 else np.log10(row.target), axis=1)
                if agregate_runs:
                    df_avg = pd.DataFrame(columns=['target'])
                    df_ids_no_run_unique = np.unique(['_'.join(i.split("_")[0:-1]) for i in df_merged.index])
                    for id in df_ids_no_run_unique:
                        variable = id+'.*'
                        id_parts = id.split('_')
                        new_id = id_parts[0] + '_' + id_parts[1]
                        df_sub = df_merged[df_merged.index.str.contains(fr'\b{variable}\b', regex=True)]
                        avg = np.median(df_sub['target'], axis = 0)
                        df_avg.loc[new_id] = avg
                    # print(df_avg.shape)
                    df_avg.to_csv(f'./data/processed_data/performance_data_regression{log_flag}/{algo}_{dim}D_{budget}.csv')                    
                else:
                    df_merged.to_csv(f'./data/processed_data/performance_data_regression{log_flag}/{algo}_{dim}D_{budget}_per_run.csv')
          
def remove_constant_value_features(df):
    return [e for e in df.columns if df[e].nunique() == 1]

def ELA_reduction(dim):
    df = pd.read_csv(f"./data/processed_data/landscape_data/ELA_LHS_{dim}D_1000n_100seed_all.csv", index_col=0)
    
    # Drop irrelevant features
    drop_col = ['limo.avg_length.reg', 'limo.avg_length.norm', 'limo.length.mean', 'limo.length.sd', 'limo.cor.reg', 'limo.cor.norm', 'limo.ratio.mean', 'limo.ratio.sd', 'limo.sd_ratio.reg', 'limo.sd_ratio.norm', 'limo.sd_mean.reg', 'limo.sd_mean.norm', 'limo.costs_fun_evals', 'limo.costs_runtime']
    df = df.drop(drop_col, axis=1)
    drop_col = []
    for col in df.columns:
        if "costs_runtime" in col or "costs_fun_evals" in col:
            drop_col.append(col)
    df = df.drop(drop_col, axis=1)
    
    # Drop features with constant values
    drop_col = remove_constant_value_features(df)
    df = df.drop(drop_col, axis=1)

    # Drop features with NaN values
    drop_col = list(df.columns[df.isnull().any()])
    df = df.drop(drop_col, axis=1)
    
    df.to_csv(f"./data/processed_data/landscape_data/ELA_LHS_{dim}D_1000n_100seed_reduced.csv")

algos = ["DE", "DiagonalCMA", "modcma", "modde", "NaiveIsoEMNA", "NGOpt14", "NGOpt38", "OnePlusOne", "PSO", "RandomSearch", "RCobyla"]
dims =  [5, 20]
budgets = [100, 250, 500, 1000, 2500, 5000, 10000]
runs = 50
# get_ng_performance_data(algos, dims, budgets, runs, False, True)
get_landscape_data()
ELA_reduction(dim=20)
ELA_reduction(dim=5)
